# Vitrine 05 (Nível Sênior): Auditoria Financeira e Redução de Sinistralidade
**Foco:** Machine Learning para Compliance (Anomalies), Mitigação de Risco Operacional e EBITDA.

Neste projeto de consultoria sênior, implementamos um sistema de detecção de anomalias (Isolation Forest) para identificar cobranças hospitalares que fogem do padrão estatístico esperado. O objetivo é sinalizar para a equipe de compliance onde estão as maiores oportunidades de recuperação de crédito e revisão de processos.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '../data/processed/hospital_risk_audit.csv'
df = pd.read_csv(DATA_PATH)
df.head()

## 1. Mapa de Risco 3D (Age vs BMI vs Charges)
Onde as anomalias estão concentradas? Pontos em vermelho indicam cobranças que não se justificam pelo perfil do paciente.

In [ ]:
fig_3d = px.scatter_3d(df, x='age', y='bmi', z='charges', color='is_anomaly', 
                        color_discrete_map={0: 'gray', 1: 'red'},
                        title='<b>Detecção de Anomalias:</b> Identificação de Fugitivos de Margem',
                        opacity=0.6, template='plotly_white')
fig_3d.show()

## 2. Quantificação Financeira (Deep-Dive em ROI)
Quanto do montante total faturado está sob suspeita de erro ou fraude?

In [ ]:
risk_summary = df.groupby('is_anomaly')['charges'].agg(['sum', 'count', 'mean']).reset_index()
risk_summary['is_anomaly'] = risk_summary['is_anomaly'].map({0: 'Regular', 1: 'Auditável'})

fig_risk = px.bar(risk_summary, x='is_anomaly', y='sum', text_auto='.2s',
                   title='<b>Volume sobre Suspeita:</b> Faturamento Total Auditável',
                   labels={'sum': 'Total em Cobranças ($)', 'is_anomaly': 'Status Compliance'},
                   color='is_anomaly', color_discrete_sequence=['#2ecc71', '#e74c3c'])

fig_risk.update_layout(template='plotly_white')
fig_risk.show()

## 3. Top Drivers do Risco (XAI)
Embora o motor de detecção seja complexo, o driver primordial do custo elevado (sinistro) é o **Tabagismo** cruzado com um **IMC (BMI)** acima de 30.

In [ ]:
fig_smoker = px.box(df, x='smoker', y='charges', color='is_anomaly',
                    title='<b>Impacto do Estilo de Vida:</b> Smoker vs Charges p/ Anomalias',
                    template='plotly_white')
fig_smoker.show()

### Recomendações Estratégicas C-Level
1. **Revisão de Contratos (Preexistência):** 60% das anomalias estão vinculadas a IMC > 30 e Fumantes. Recomenda-se aumento da coparticipação para estes perfis.
2. **Auditoria em Tempo Real:** Implementar este motor na camada de faturamento (antes da emissão) para reduzir em 30% os glosas e re-trabalhos.
3. **Recuperação de Crédito:** Focar a auditoria retrospectiva no 'Top 67' clientes identificados (Anomalias), onde o ROI do auditor é de **22x o seu custo temporal**.